[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/gauravs19/iiot-predictive-maintenance/blob/main/notebooks/00_setup_and_data.ipynb)

# 00 · Setup & Data

**Goal of this notebook:** get a clean, reproducible starting point. By the end you
will have both open datasets downloaded, validated, and understood at a high level.

This project demonstrates two classic Industrial-IoT / manufacturing ML problems:

| Problem | Question it answers | Dataset | Notebook |
|---|---|---|---|
| **Predictive maintenance** | *Will this machine fail, and how soon?* | AI4I 2020 + NASA C-MAPSS | `02` |
| **Anomaly detection** | *Is this machine behaving abnormally right now?* | NASA C-MAPSS | `03` |

**Why these two datasets?**
- **AI4I 2020** is small, tabular, and *labelled* — perfect for a supervised
  "warm-up": predict a failure flag from a single snapshot of sensor readings.
- **NASA C-MAPSS** is *run-to-failure time-series* data from simulated turbofan
  engines — the canonical benchmark for **Remaining-Useful-Life (RUL)** estimation
  and a realistic setting for **unsupervised** anomaly detection.

Everything downloads at runtime, so nothing large is stored in git.

## Step 1 · Bootstrap the environment

The cell below does three things, and is designed to run **identically** on your
laptop and on Google Colab:

1. Detects whether we're on Colab. If so, it clones this repo and `pip install`s
   the dependencies (Colab starts from a blank machine each session).
2. Walks up the folder tree to find the **repo root** (the folder containing
   `src/`) and adds it to `sys.path`. That's what lets `from src import data`
   work even though the notebook lives inside `notebooks/`.
3. Prints where it's running so you can confirm the setup.

> All four notebooks start with this same cell.

In [ ]:
# --- Environment bootstrap (works locally AND on Google Colab) ---------------
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # On Colab there is no repo yet, so clone it and install dependencies.
    !git clone -q https://github.com/gauravs19/iiot-predictive-maintenance.git
    %cd iiot-predictive-maintenance
    !pip install -q -r requirements.txt

# Make the repo root importable so `from src import ...` works from notebooks/.
def _find_repo_root(start="."):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, "src")):
            return p
        p = os.path.dirname(p)
    raise RuntimeError("repo root (folder containing src/) not found")

REPO_ROOT = _find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)
print("running on Colab" if IN_COLAB else "running locally")

## Step 2 · Imports

We import our own `src` package (the reusable pipeline code) plus the standard
data-science stack. Keeping the heavy logic in `src/` rather than in the notebook
means the notebooks stay short and readable, and the same code is unit-testable
and reusable by the sibling `iiot-ai-rag` project later.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 40)

from src import data   # our data-loading module (src/data.py)
print("imports OK")

## Step 3 · Load the AI4I 2020 dataset

`data.load_ai4i()` fetches the dataset straight from the UCI Machine Learning
Repository (via the `ucimlrepo` package, with a CSV-download fallback) and returns
a single tidy `DataFrame`.

**What the data represents:** 10,000 rows, each a snapshot of one synthetic milling
machine. Columns include process parameters (air & process temperature, rotational
speed, torque, tool wear) and a `Machine failure` flag plus five specific
failure-mode flags (tool wear failure, heat dissipation, power, overstrain, random).

We display the shape and the first rows to confirm it loaded correctly.

In [ ]:
ai4i = data.load_ai4i()
print("AI4I shape:", ai4i.shape)
ai4i.head()

### Inspect the failure balance

Real machines fail *rarely*, and this dataset reflects that — only ~3.4% of rows
are failures. This **class imbalance** is important: it means accuracy alone is a
misleading metric (a model predicting "never fails" would be ~96.6% accurate but
useless). We'll come back to this in notebook `02` by using precision/recall and
class weighting.

In [ ]:
print("Failure rate: %.2f%%" % (100 * ai4i["Machine failure"].mean()))
ai4i["Machine failure"].value_counts()

## Step 4 · Load the NASA C-MAPSS dataset (FD001)

`data.load_cmapss("FD001")` downloads three text files from a community mirror and
returns a dict with three DataFrames:

- **`train`** — 100 engines run from healthy all the way **to failure**. Each row is
  one operational cycle.
- **`test`** — 100 *different* engines, but the series are **truncated** some time
  before failure. The model must estimate how much life remains.
- **`rul`** — the ground-truth remaining cycles for each test engine's final row
  (used only to score predictions).

Each row has: an engine `unit` id, the `cycle` number, 3 operational settings, and
21 sensor measurements (temperatures, pressures, speeds, flow ratios, etc.).

In [ ]:
cmapss = data.load_cmapss("FD001")
for k, v in cmapss.items():
    print(f"{k:6s} shape: {v.shape}")
cmapss["train"].head()

### How long does each engine survive?

A quick sanity check: in the training set every engine runs to failure, so the max
cycle per unit is its lifetime. The spread below shows engines fail at very
different ages (≈128 to ≈360 cycles) — exactly the variability that makes RUL
prediction a real problem rather than a fixed schedule.

In [ ]:
lifetimes = cmapss["train"].groupby("unit")["cycle"].max()
print("Engine lifetimes — min: %d  median: %d  max: %d"
      % (lifetimes.min(), lifetimes.median(), lifetimes.max()))
lifetimes.describe().round(1)

## Step 5 · Where this fits the IIoT reference architecture

This notebook is the **data ingestion** layer. Mapping the project onto a typical
Industrial-IoT stack:

```
 Edge sensors ─▶ Ingest/Store ─▶ Feature engineering ─▶ ML models ─▶ Serving/Action
 (turbofan,        (this nb:        (notebook 01)        (nb 02/03)    (future:
  milling)          load + cache)                                      iiot-ai-rag)
```

**Next:** notebook `01` explores the sensors visually and engineers the features
both model families will consume.